# Set up

In [1]:
# Imports
import json
from pathlib import Path
from functools import partial
import os
from dotenv import load_dotenv
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from datasets import Dataset, DatasetDict
from sklearn.model_selection import train_test_split
from torch.utils.data import DataLoader
from transformers import AutoModel, AutoTokenizer

In [ ]:
# Set root constant, checking if running in Colab
try:
    from google.colab import drive
    drive.mount("/content/drive", force_remount=False)
    ROOT = Path(os.environ.get("COLAB_PROJECT_ROOT", "/content/drive/MyDrive/Scrubs"))
except ImportError:
    load_dotenv()
    ROOT = Path(os.environ.get("PROJECT_ROOT"))

# Constants
LABELED_SCENES_PATH = ROOT / "data" / "labeled_scenes.json"
MODELS_DIR = ROOT / "DeBERTa" / "models"
OUTPUT_DIR = ROOT / "data" / "DeBERTa_predictions" / "labeled_scenes"
BATCH_SIZE = 8
MODEL_NAMES = ["microsoft/deberta-v3-small", "microsoft/deberta-v3-base"]
OPTIMIZERS = ["SGD", "AdamW"]
DROPOUT_RATE = 0.01
ADAM_LEARNING_RATE = 2e-6
SGD_LEARNING_RATE = 0.0001
NUM_EPOCHS = 25
MAX_NORM = 1.0
RANDOM_SEED = 100

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


# Load and split data to create collate function, and dataloaders

In [3]:
def load_data() -> pd.DataFrame:
    """
    Return labeled scenes JSON as dataframe
    """
    with open(LABELED_SCENES_PATH, "r") as f:
        labeled_scenes = json.load(f)

    return pd.DataFrame(labeled_scenes)


def split_data(data_df: pd.DataFrame) -> DatasetDict:
    """
    Split data into train, validate, and test sets based on episode,
    returning a DatasetDict
    """

    # Create array of unique episode IDs
    episodes = np.asarray(data_df["episode_id"].unique())

    # Split episodes into train/test set
    episode_splits = {}
    episode_splits["train"], train_test = train_test_split(
        episodes, test_size=0.3, random_state=RANDOM_SEED
    )

    # Split test further into train/validate
    episode_splits["validate"], episode_splits["test"] = train_test_split(
        train_test, test_size=0.5, random_state=RANDOM_SEED
    )

    # Use episode assignments to actually split data
    data_splits = {}
    for subset in ["train", "validate", "test"]:
        data_splits[subset] = (
            data_df[data_df["episode_id"].isin(episode_splits[subset])]
            .reset_index(drop=True)  # drop index
        )

    # Create DatasetDict with splits
    dataset_dict = DatasetDict({
        "train": Dataset.from_pandas(data_splits["train"]),
        "validate": Dataset.from_pandas(data_splits["validate"]),
        "test": Dataset.from_pandas(data_splits["test"]),
    })
    return dataset_dict


def collate(batch: list[dict], tokenizer: AutoTokenizer):
    """
    Collate function to convert batch of scenes into tensors.
    DataLoader only passes (batch); bind tokenizer via partial when creating the DataLoader.
    """
    # Convert sad and funny labels to tensors
    labels = torch.tensor(
        [[float(scene["funny"]), float(scene["sad"])] for scene in batch],
        dtype=torch.float32,
    )

    # Convert scene positions to tensor
    positions = torch.tensor(
        [float(scene["position"]) for scene in batch],
        dtype=torch.float32,
    ).unsqueeze(1)  # needed to concatenate with scene encodings

    # Get embeddings for previous scene text
    prev_texts = [scene["prev_scene_text"] for scene in batch]
    prev_tokens = tokenizer(
        prev_texts,
        padding=True,
        truncation=True,
        max_length=512,
        return_tensors="pt",
    )

    # Get embeddings for current scene text
    curr_texts = [scene["text"] for scene in batch]
    curr_tokens = tokenizer(
        curr_texts,
        padding=True,
        truncation=True,
        max_length=512,
        return_tensors="pt",
    )

    # Also get scene metadata to use for inference
    metadata = [
        {
            "episode_id": scene["episode_id"],
            "scene_id": scene["scene_id"],
            "position": scene["position"],
            "prev_scene_text": scene["prev_scene_text"],
            "text": scene["text"],
        }
    for scene in batch
]

    return prev_tokens, curr_tokens, positions, labels, metadata


def create_dataloaders(data: DatasetDict, tokenizer: AutoTokenizer) -> tuple:
    """
    Create dataloaders for train, validate, and test sets,
    returning a tuple of DataLoaders for train, validate, and test sets.
    """

    # Set a collate function that contains the tokenizer
    collate_fn = partial(collate, tokenizer=tokenizer)

    train_dataloader = DataLoader(
        data["train"],
        batch_size=BATCH_SIZE,
        shuffle=True,
        collate_fn=collate_fn,
    )
    validate_dataloader = DataLoader(
        data["validate"],
        batch_size=BATCH_SIZE,
        shuffle=False,
        collate_fn=collate_fn,
    )
    test_dataloader = DataLoader(
        data["test"],
        batch_size=BATCH_SIZE,
        shuffle=False,
        collate_fn=collate_fn,
    )
    return train_dataloader, validate_dataloader, test_dataloader

# Set up classes for DeBERTa Model and Trainer

In [ ]:
# Scrubs DeBERTa Model
class ScrubsDeBERTa(nn.Module):
    def __init__(self, model_name: str, dropout: bool):
        super().__init__()

        # Load pretrained model
        self.pretrained_model = AutoModel.from_pretrained(model_name)

        # Add dropout layer to avoid overfitting if specified in input
        self.dropout = nn.Dropout(DROPOUT_RATE) if dropout else None

        # Input dimension for classification heads is hidden_dim * 3 + 1.
        # hidden_dim * 3 because of embeddings for previous scene text, current
        # scene text, and difference between the two.
        # +1 for the position of the scene in the episode.
        hidden_dim = self.pretrained_model.config.hidden_size
        input_dim = hidden_dim * 3 + 1

        # Output dimension for classification heads is num of rating classes: 5
        output_dim = 5

        # Create classification heads for funny and sad ratings
        self.funny_head = nn.Linear(input_dim, output_dim)
        self.sad_head = nn.Linear(input_dim, output_dim)

    def forward(
        self,
        prev_input_ids,
        prev_attention_mask,
        curr_input_ids,
        curr_attention_mask,
        positions,
        labels=None,
        **kwargs,
    ):
        # Previous scene encoding, masking padded tokens
        prev_scene_outputs = self.pretrained_model(
            prev_input_ids,
            attention_mask=prev_attention_mask,
        )

        # Either get embedding from CLS token or avg across all tokens
        # Source: https://discuss.huggingface.co/t/common-practice-using-the-hidden-state-associated-with-cls-as-an-input-feature-for-a-classification-task/14003
        if self.encoding_type == "cls":
            prev_scene_encoding = prev_scene_outputs.last_hidden_state[:, 0, :]
        else: # mean pooling
            prev_scene_encoding = prev_scene_outputs.last_hidden_state.mean(dim=1)

        # Current scene encoding, masking padded tokens
        scene_outputs = self.pretrained_model(
            curr_input_ids,
            attention_mask=curr_attention_mask,
        )

        # Either get embedding from CLS token or avg across all tokens
        if self.encoding_type == "cls":
            scene_encoding = scene_outputs.last_hidden_state[:, 0, :]
        else: # mean pooling
            scene_encoding = scene_outputs.last_hidden_state.mean(dim=1)

        # Calculate difference between previous scene and current scene
        diff = scene_encoding - prev_scene_encoding

        # Concatenate encodings + position
        combined = torch.cat(
            [prev_scene_encoding, scene_encoding, diff, positions], dim=1
        )

        # Perform dropout
        if self.dropout is not None:
            combined = self.dropout(combined)

        # Get logits from each head (size of batch, 5)
        funny_logits = self.funny_head(combined)
        sad_logits = self.sad_head(combined)

        forward_rv = {"logits": (funny_logits, sad_logits)}

        # If labels provided, calculate loss
        if labels is not None:
            # Convert to 0-4 for cross entropy loss
            funny_labels = labels[:, 0].long() - 1
            sad_labels = labels[:, 1].long() - 1

            # Calculate loss for each head
            loss_funny = F.cross_entropy(funny_logits, funny_labels)
            loss_sad = F.cross_entropy(sad_logits, sad_labels)

            # Add loss for each head
            forward_rv["loss"] = loss_funny + loss_sad

        return forward_rv


class ScrubsDeBERTaTrainer:
    def __init__(
        self,
        model: nn.Module,
        model_name: str,
        train_dataloader: DataLoader,
        validate_dataloader: DataLoader,
        optimizer: str,
        encoding_type: str,

    ):
        self.model = model

        # Type of encoding (using CLS token or mean pooling)
        self.model.encoding_type = encoding_type

        # Dataloaders
        self.train_loader = train_dataloader
        self.validate_loader = validate_dataloader

        # Convert model to CUDA if needed
        self.device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
        self.model.to(self.device)

        # Use SGD or AdamW optimizer as specified
        if optimizer == "SGD":
            self.optimizer = torch.optim.SGD(
                self.model.parameters(),
                lr=SGD_LEARNING_RATE,
            )
        else: # AdamW
            self.optimizer = torch.optim.AdamW(
                self.model.parameters(),
                lr=ADAM_LEARNING_RATE,
            )


        # Best model tracking for using later with inference
        self.best_val_loss = float("inf")

        # Best model path
        os.makedirs(MODELS_DIR, exist_ok=True)
        dropout_str = "with_dropout" if self.model.dropout else "without_dropout"
        model_str = model_name.replace("-", "_").replace("/", "_")
        self.best_model_path = MODELS_DIR / f"{model_str}_{optimizer}_{dropout_str}_best_model.pt"
        self.model.best_model_path = self.best_model_path

    def train_epoch(self):
        self.model.train()
        total_loss = 0

        # Iterate over batches of scenes from data loader (ignore metadata)
        for prev_tokens, curr_tokens, positions, labels, _ in self.train_loader:
            # Convert for GPU if needed
            prev_tokens = {k: v.to(self.device) for k, v in prev_tokens.items()}
            curr_tokens = {k: v.to(self.device) for k, v in curr_tokens.items()}
            positions = positions.to(self.device)
            labels = labels.to(self.device)

            # Reset gradients
            self.optimizer.zero_grad()

            # Forward pass
            outputs = self.model(
                prev_input_ids=prev_tokens["input_ids"],
                prev_attention_mask=prev_tokens["attention_mask"],
                curr_input_ids=curr_tokens["input_ids"],
                curr_attention_mask=curr_tokens["attention_mask"],
                positions=positions,
                labels=labels,
            )
            loss = outputs["loss"]

            # Backward pass
            loss.backward()

            # Update weights
            self.optimizer.step()

            # Add to loss
            total_loss += loss.item()

        return total_loss / len(self.train_loader)

    def evaluate(self):
        self.model.eval()
        total_loss = 0

        # Don't save gradients since not needed for validation
        with torch.no_grad():
            # Iterate over batches of scenes from data loader (ignore metadata)
            for prev_tokens, curr_tokens, positions, labels, _ in self.validate_loader:
                # Set labels and inputs for GPU compatability if needed
                prev_tokens = {k: v.to(self.device) for k, v in prev_tokens.items()}
                curr_tokens = {k: v.to(self.device) for k, v in curr_tokens.items()}
                positions = positions.to(self.device)
                labels = labels.to(self.device)

                # Pass inputs to model from output of tokenizer
                outputs = self.model(
                    prev_input_ids=prev_tokens["input_ids"],
                    prev_attention_mask=prev_tokens["attention_mask"],
                    curr_input_ids=curr_tokens["input_ids"],
                    curr_attention_mask=curr_tokens["attention_mask"],
                    positions=positions,
                    labels=labels,
                )

                # Calculate loss add to total
                total_loss += outputs["loss"].item()

        return total_loss / len(self.validate_loader)

    def save_if_best(self, val_loss):
        # If loss is smaller than current best loss, save model for later use
        if val_loss < self.best_val_loss:
            self.best_val_loss = val_loss
            torch.save(self.model.state_dict(), self.best_model_path)

# Functions to run training and inference

In [ ]:
def run_training(   model_name: str,
                    encoding_type: str,
                    optimizer: str, # SGD or AdamW
                    dropout: bool, # with (True) or without (False) dropout
                    train_dataloader: DataLoader,
                    validate_dataloader: DataLoader,
):
    """
    Train the model using the test set
    """
    # Initialize model
    model = ScrubsDeBERTa(
        model_name=model_name,
        dropout=dropout)

    # Convert to GPU if available
    model.float()
    model.to("cuda" if torch.cuda.is_available() else "cpu")

    # Initialize trainer
    trainer = ScrubsDeBERTaTrainer(
        model=model,
        model_name=model_name,
        train_dataloader=train_dataloader,
        validate_dataloader=validate_dataloader,
        optimizer=optimizer,
        encoding_type=encoding_type,
    )

    # Report loss per epoch
    for epoch in range(NUM_EPOCHS):
        train_loss = trainer.train_epoch()
        val_loss = trainer.evaluate()
        trainer.save_if_best(val_loss)
        print(
            f"{model_name} - optimizer: {optimizer} - dropout: {dropout} - "
            f"encoding type: {encoding_type} - epoch {epoch + 1}"
            f": training loss = {train_loss}, validation loss = {val_loss} "
        )

    return model, trainer


def run_inference(model, trainer, test_dataloader) -> list[dict]:
    """
    Run inference on test set, returning list of dictionaries with labels
    and predicted ratings for each scene
    """

    # Load best model
    model.load_state_dict(torch.load(trainer.best_model_path))
    model.eval()

    # Device to use for inference
    device = next(model.parameters()).device

    # Iterate over batches of scenes from data loader, storing results in list
    results = []
    with torch.no_grad():
        for prev_tokens, curr_tokens, positions, labels_batch, metadata in test_dataloader:

            # Convert to GPU if needed
            prev_tokens = {k: v.to(device) for k, v in prev_tokens.items()}
            curr_tokens = {k: v.to(device) for k, v in curr_tokens.items()}
            positions = positions.to(device)
            labels_batch = labels_batch.to(device)

            # Forward pass
            result = model(
                prev_input_ids=prev_tokens["input_ids"],
                prev_attention_mask=prev_tokens["attention_mask"],
                curr_input_ids=curr_tokens["input_ids"],
                curr_attention_mask=curr_tokens["attention_mask"],
                positions=positions,
            )

            # Unpack logits tuple
            funny_logits, sad_logits = result["logits"]

            # Convert to list of integers
            pred_funny = (funny_logits.argmax(dim=-1) + 1).cpu().tolist()
            pred_sad = (sad_logits.argmax(dim=-1) + 1).cpu().tolist()
            labels = labels_batch.cpu().tolist()

            # Append results for each scene
            for scene_idx in range(len(labels)):
                results.append({
                    "episode_id": metadata[scene_idx]["episode_id"],
                    "scene_id": metadata[scene_idx]["scene_id"],
                    "position": metadata[scene_idx]["position"],
                    "prev_scene_text": metadata[scene_idx]["prev_scene_text"],
                    "text": metadata[scene_idx]["text"],
                    "true_funny": int(labels[scene_idx][0]),
                    "true_sad": int(labels[scene_idx][1]),
                    "predicted_funny": pred_funny[scene_idx],
                    "predicted_sad": pred_sad[scene_idx],
                })

    return results

# Main function
## Run training and inference for each combination of parameters:
 ### - DeBERTa-v3-small vs. DeBERTa-v3-base
 ### - Adam optimizer with decay vs. SGD
 ### - Dropout applied vs. not
 ### - Use CLS token to interpret DeBERTa encodings vs. mean pooling

In [ ]:
def main():

    # Load data
    data = split_data(load_data())

    # Create directory for output
    os.makedirs(OUTPUT_DIR, exist_ok=True)
    
    for model_name in MODEL_NAMES:

        # Get model-specific tokenizer and dataloaders
        tokenizer = AutoTokenizer.from_pretrained(model_name)
        train_loader, valid_loader, test_loader = create_dataloaders(data, tokenizer)

        # Iterate over combinations of optimizer type, dropout boolean, and text encoding type
        for optimizer in OPTIMIZERS:
            for dropout in [True, False]:
                for encoding_type in ["cls", "mean_pooling"]:


                    # Run training
                    model, trainer = run_training(
                        encoding_type=encoding_type,
                        model_name=model_name,
                        optimizer=optimizer,
                        dropout=dropout,
                        train_dataloader=train_loader,
                        validate_dataloader=valid_loader,
                    )

                    # Run inference
                    results = run_inference(model, trainer, test_loader)

                    # Fix parts of name for output
                    model_str = model_name.replace("-", "_").replace("/", "_")
                    dropout_str = "with_dropout" if dropout else "without_dropout"

                    # Concatenate file path
                    name_parts = [model_str, optimizer, dropout_str, encoding_type]
                    path = OUTPUT_DIR / ("_".join(name_parts) + "_results.json")

                    # Actual export
                    with open(path , "w") as f:
                        json.dump(results, f)

if __name__ == "__main__":
    main()

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


Loading weights:   0%|          | 0/102 [00:00<?, ?it/s]

DebertaV2Model LOAD REPORT from: microsoft/deberta-v3-small
Key                                     | Status     |  | 
----------------------------------------+------------+--+-
mask_predictions.dense.weight           | UNEXPECTED |  | 
mask_predictions.LayerNorm.weight       | UNEXPECTED |  | 
lm_predictions.lm_head.LayerNorm.bias   | UNEXPECTED |  | 
lm_predictions.lm_head.dense.weight     | UNEXPECTED |  | 
mask_predictions.classifier.weight      | UNEXPECTED |  | 
mask_predictions.classifier.bias        | UNEXPECTED |  | 
lm_predictions.lm_head.LayerNorm.weight | UNEXPECTED |  | 
lm_predictions.lm_head.dense.bias       | UNEXPECTED |  | 
mask_predictions.LayerNorm.bias         | UNEXPECTED |  | 
mask_predictions.dense.bias             | UNEXPECTED |  | 
lm_predictions.lm_head.bias             | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


microsoft/deberta-v3-small - optimizer: SGD - dropout: True - encoding type: cls - epoch 1: training loss = 3.086212544441223, validation loss = 2.8237393697102866 
microsoft/deberta-v3-small - optimizer: SGD - dropout: True - encoding type: cls - epoch 2: training loss = 2.7354836225509644, validation loss = 2.697254366344876 
microsoft/deberta-v3-small - optimizer: SGD - dropout: True - encoding type: cls - epoch 3: training loss = 2.624484395980835, validation loss = 2.6308169894748263 
microsoft/deberta-v3-small - optimizer: SGD - dropout: True - encoding type: cls - epoch 4: training loss = 2.5655855894088746, validation loss = 2.6044308609432645 
microsoft/deberta-v3-small - optimizer: SGD - dropout: True - encoding type: cls - epoch 5: training loss = 2.5651813745498657, validation loss = 2.612380954954359 
microsoft/deberta-v3-small - optimizer: SGD - dropout: True - encoding type: cls - epoch 6: training loss = 2.493248324394226, validation loss = 2.59861069255405 
microsoft/d

Loading weights:   0%|          | 0/102 [00:00<?, ?it/s]

...

DebertaV2Model LOAD REPORT from: microsoft/deberta-v3-base
Key                                     | Status     |  | 
----------------------------------------+------------+--+-
mask_predictions.dense.weight           | UNEXPECTED |  | 
mask_predictions.LayerNorm.weight       | UNEXPECTED |  | 
lm_predictions.lm_head.LayerNorm.bias   | UNEXPECTED |  | 
lm_predictions.lm_head.dense.weight     | UNEXPECTED |  | 
mask_predictions.classifier.weight      | UNEXPECTED |  | 
mask_predictions.classifier.bias        | UNEXPECTED |  | 
lm_predictions.lm_head.LayerNorm.weight | UNEXPECTED |  | 
lm_predictions.lm_head.dense.bias       | UNEXPECTED |  | 
mask_predictions.LayerNorm.bias         | UNEXPECTED |  | 
mask_predictions.dense.bias             | UNEXPECTED |  | 
lm_predictions.lm_head.bias             | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


microsoft/deberta-v3-base - optimizer: AdamW - dropout: False - encoding type: cls - epoch 1: training loss = 2.8343015766143798, validation loss = 2.5497875213623047 
microsoft/deberta-v3-base - optimizer: AdamW - dropout: False - encoding type: cls - epoch 2: training loss = 2.607556245326996, validation loss = 2.536640829510159 
microsoft/deberta-v3-base - optimizer: AdamW - dropout: False - encoding type: cls - epoch 3: training loss = 2.560538101196289, validation loss = 2.522627353668213 
microsoft/deberta-v3-base - optimizer: AdamW - dropout: False - encoding type: cls - epoch 4: training loss = 2.56012318611145, validation loss = 2.5027091238233776 
microsoft/deberta-v3-base - optimizer: AdamW - dropout: False - encoding type: cls - epoch 5: training loss = 2.4985064172744753, validation loss = 2.5156462457444935 
microsoft/deberta-v3-base - optimizer: AdamW - dropout: False - encoding type: cls - epoch 6: training loss = 2.5198991918563842, validation loss = 2.5523940457238092

Loading weights:   0%|          | 0/198 [00:00<?, ?it/s]

DebertaV2Model LOAD REPORT from: microsoft/deberta-v3-base
Key                                     | Status     |  | 
----------------------------------------+------------+--+-
mask_predictions.dense.weight           | UNEXPECTED |  | 
mask_predictions.LayerNorm.weight       | UNEXPECTED |  | 
lm_predictions.lm_head.LayerNorm.bias   | UNEXPECTED |  | 
lm_predictions.lm_head.dense.weight     | UNEXPECTED |  | 
mask_predictions.classifier.weight      | UNEXPECTED |  | 
mask_predictions.classifier.bias        | UNEXPECTED |  | 
lm_predictions.lm_head.LayerNorm.weight | UNEXPECTED |  | 
lm_predictions.lm_head.dense.bias       | UNEXPECTED |  | 
mask_predictions.LayerNorm.bias         | UNEXPECTED |  | 
mask_predictions.dense.bias             | UNEXPECTED |  | 
lm_predictions.lm_head.bias             | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


microsoft/deberta-v3-base - optimizer: AdamW - dropout: False - encoding type: mean_pooling - epoch 1: training loss = 2.807106432914734, validation loss = 2.6625679069095187 
microsoft/deberta-v3-base - optimizer: AdamW - dropout: False - encoding type: mean_pooling - epoch 2: training loss = 2.5552344703674317, validation loss = 2.596137947506375 
microsoft/deberta-v3-base - optimizer: AdamW - dropout: False - encoding type: mean_pooling - epoch 3: training loss = 2.5306915807724, validation loss = 2.5653654999203153 
microsoft/deberta-v3-base - optimizer: AdamW - dropout: False - encoding type: mean_pooling - epoch 4: training loss = 2.497384922504425, validation loss = 2.6014725632137723 
microsoft/deberta-v3-base - optimizer: AdamW - dropout: False - encoding type: mean_pooling - epoch 5: training loss = 2.482717957496643, validation loss = 2.5303542216618857 
microsoft/deberta-v3-base - optimizer: AdamW - dropout: False - encoding type: mean_pooling - epoch 6: training loss = 2.4